In [ ]:
# XGBoost — hand-crafted time-series features (WORKSHOP VERSION, self-contained)
#
# Role in the workshop: this is the "general / basic preprocessing" track — plain
# statistical + frequency-domain features computed directly on the raw 20-channel
# EEG signal. No bipolar montage, no mu-law encoding, no other EEG-specific tricks.
# (Compare against the 1D CNN notebook, which uses the same raw signal but adds
# those domain-specific preprocessing steps — that's where "domain knowledge" is
# meant to show up in this workshop.)
#
# Features per channel (7 total, x 19 channels = 133-dim feature vector):
#   mean, variance, zero-crossing rate, delta/theta/alpha/beta band power
#
# No sample weighting: the workshop subset is already class-balanced
# (100 rows/class train, 20 rows/class val), so there's no imbalance to correct
# for. (Full-scale training on the real, imbalanced ~5,900-row clean dataset is
# where sample weighting / 2-step training matter — see the canonical pipeline
# in kaggle_upload/src for that version.)
#
# Fully self-contained — no external .py imports.

In [ ]:
import os, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
import xgboost as xgb
from sklearn.metrics import f1_score

IS_KAGGLE = os.path.exists('/kaggle')
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | xgboost {xgb.__version__}")

In [ ]:
# ============ Config ============
SEED       = 42
FS         = 200          # Hz
WINDOW_LEN = 10_000        # samples = 50 s at 200 Hz

XGB_PARAMS = {
    "objective"        : "multi:softprob",
    "num_class"        : 6,
    "eval_metric"      : "mlogloss",
    "tree_method"      : "hist",
    "max_depth"        : 6,
    "learning_rate"    : 0.05,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "min_child_weight" : 5,
    "seed"             : SEED,
}
NUM_BOOST_ROUND        = 500
EARLY_STOPPING_ROUNDS  = 30

CHANNELS    = ['Fp1','F3','C3','P3','F7','T3','T5','O1','Fz','Cz','Pz',
               'Fp2','F4','C4','P4','F8','T4','T6','O2']  # 19 EEG channels (EKG excluded)
TARGETS     = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
CLASS_NAMES = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']

random.seed(SEED)
np.random.seed(SEED)

if IS_KAGGLE:
    DATA_ROOT       = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    RAW_TRAIN_PATH  = os.path.join(DATA_ROOT, 'train.csv')
    EEG_DIR         = os.path.join(DATA_ROOT, 'train_eegs')
    SAMPLE_IDS_PATH = '/kaggle/input/datasets/xiaosufrankhu/midas-summer-academy-wk3-eeg/workshop_sample_ids.csv'
    EEG_CACHE_DIR   = '/kaggle/working/eeg_cache_xgb'
else:
    DATA_ROOT       = os.path.abspath('../')
    RAW_TRAIN_PATH  = os.path.abspath('../data_raw/train.csv')
    EEG_DIR         = os.path.join(DATA_ROOT, 'train_eegs')
    SAMPLE_IDS_PATH = os.path.abspath('../data_raw/workshop_sample_ids.csv')
    EEG_CACHE_DIR   = os.path.abspath('../eeg_cache_xgb')

# NOTE: if running on Kaggle and this path doesn't exist, run !ls /kaggle/input
# and update DATA_ROOT / EEG_DIR to match how the competition data was attached.
print(f"Competition data path exists: {os.path.exists(DATA_ROOT)}")

In [ ]:
# ============ Data loading (workshop subset) ============
raw_df     = pd.read_csv(RAW_TRAIN_PATH)
sample_ids = pd.read_csv(SAMPLE_IDS_PATH)

merged = raw_df.merge(
    sample_ids[["eeg_id", "eeg_sub_id", "split"]],
    on=["eeg_id", "eeg_sub_id"],
    how="inner",
)
train_df = merged[merged["split"] == "train"].reset_index(drop=True)
val_df   = merged[merged["split"] == "val"].reset_index(drop=True)

print(f'Train: {len(train_df):,} rows')
print(f'Val   : {len(val_df):,} rows')
print(train_df['expert_consensus'].value_counts())

In [ ]:
# ============ Raw EEG caching ============
# Several rows may share the same eeg_id but use a different label window
# (eeg_label_offset_seconds), so we cache each eeg_id's full raw recording once.

os.makedirs(EEG_CACHE_DIR, exist_ok=True)

def cache_one_eeg(eeg_id: int, eeg_dir: str, cache_dir: str) -> None:
    dst = os.path.join(cache_dir, f"{eeg_id}.npy")
    if os.path.exists(dst):
        return
    src = os.path.join(eeg_dir, f"{eeg_id}.parquet")
    eeg = pd.read_parquet(src, columns=CHANNELS).to_numpy(dtype=np.float32)  # (T, 19)
    np.save(dst, eeg)

needed_eeg_ids = set(train_df['eeg_id']).union(val_df['eeg_id'])
for eeg_id in needed_eeg_ids:
    cache_one_eeg(int(eeg_id), EEG_DIR, EEG_CACHE_DIR)

print(f'Cache ready: {len(needed_eeg_ids)} raw EEG recordings for workshop subset')

In [ ]:
# ============ Feature extraction (general / basic — no montage, no mu-law) ============
# 7 features x 19 raw channels = 133-dim feature vector per row.

_FREQS = np.fft.rfftfreq(WINDOW_LEN, d=1.0 / FS)
_DELTA = (_FREQS >= 0.5)  & (_FREQS <  4.0)
_THETA = (_FREQS >= 4.0)  & (_FREQS <  8.0)
_ALPHA = (_FREQS >= 8.0)  & (_FREQS < 13.0)
_BETA  = (_FREQS >= 13.0) & (_FREQS < 30.0)


def zero_crossing_rate(sig: np.ndarray) -> np.ndarray:
    """sig: (T, C) -> (C,). Fraction of adjacent-sample sign changes."""
    signs = np.sign(sig)
    signs[signs == 0] = 1  # treat exact zero as positive, avoids spurious crossings
    crossings = (np.diff(signs, axis=0) != 0).sum(axis=0)
    return crossings / (len(sig) - 1)


def extract_features(window: np.ndarray) -> np.ndarray:
    """window: (T, 19) raw, unmontaged EEG samples -> (133,) feature vector."""
    window = np.nan_to_num(window, nan=0.0, posinf=0.0, neginf=0.0)

    feat_mean = window.mean(axis=0)                                   # (19,)
    feat_var  = window.var(axis=0)                                    # (19,)
    feat_zcr  = zero_crossing_rate(window)                             # (19,)

    fft_power = (np.abs(np.fft.rfft(window, axis=0)) ** 2) / len(window)  # (F, 19)
    feat_delta = fft_power[_DELTA].sum(axis=0)
    feat_theta = fft_power[_THETA].sum(axis=0)
    feat_alpha = fft_power[_ALPHA].sum(axis=0)
    # (beta band folded in as a 4th band-power feature, matching "band power" in the design doc)
    feat_beta  = fft_power[_BETA].sum(axis=0)

    return np.concatenate([
        feat_mean, feat_var, feat_zcr, feat_delta, feat_theta, feat_alpha, feat_beta
    ])  # 7 x 19 = 133 features


def build_features(df: pd.DataFrame, cache_dir: str, window_len: int = WINDOW_LEN):
    X, y_hard, y_soft = [], [], []
    for _, row in df.iterrows():
        eeg_id = int(row['eeg_id'])
        offset = int(row['eeg_label_offset_seconds']) * FS

        eeg = np.load(os.path.join(cache_dir, f"{eeg_id}.npy"))  # (T, 19)
        window = eeg[offset:offset + window_len]
        if len(window) < window_len:
            pad_top = window_len - len(window)
            window  = np.pad(window, ((0, pad_top), (0, 0)), mode="constant")

        feats = extract_features(window)
        soft  = row[TARGETS].to_numpy(dtype=np.float32)
        soft  = soft / soft.sum() if soft.sum() > 0 else np.full(6, 1 / 6)

        X.append(feats)
        y_hard.append(int(np.argmax(soft)))
        y_soft.append(soft)

    return (np.array(X, dtype=np.float32),
            np.array(y_hard),
            np.array(y_soft, dtype=np.float32))


t0 = time.time()
X_train, y_train, y_train_soft = build_features(train_df, EEG_CACHE_DIR)
X_val,   y_val,   y_val_soft   = build_features(val_df,   EEG_CACHE_DIR)
print(f"X_train {X_train.shape} | X_val {X_val.shape}  [{time.time()-t0:.0f}s]")

In [ ]:
# ============ Train XGBoost ============
dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)

evals_result = {}
model = xgb.train(
    params                = XGB_PARAMS,
    dtrain                = dtrain,
    num_boost_round       = NUM_BOOST_ROUND,
    evals                 = [(dtrain, "train"), (dval, "val")],
    early_stopping_rounds = EARLY_STOPPING_ROUNDS,
    evals_result          = evals_result,
    verbose_eval          = 25,
)

n_rounds_trained = len(evals_result["val"]["mlogloss"])
print(f"\nRounds trained         : {n_rounds_trained}")
print(f"Best round (mlogloss)  : {model.best_iteration}  (val mlogloss={model.best_score:.4f})")

# ---- Stage val KL / F1 every 5 rounds ----
# mlogloss is a convenient proxy XGBoost can optimize round-by-round, but KL divergence
# is the metric we actually care about. They usually track each other but are not
# guaranteed to peak at the same round — staging KL/F1 lets us pick the round that is
# best on the metric that matters, and shows learners when/why the two disagree.
STAGE_EVERY  = 5
stage_rounds = list(range(STAGE_EVERY, n_rounds_trained + 1, STAGE_EVERY))
if not stage_rounds or stage_rounds[-1] != n_rounds_trained:
    stage_rounds.append(n_rounds_trained)

stage_kl, stage_f1 = [], []
for r in stage_rounds:
    prob_r = model.predict(dval, iteration_range=(0, r)).reshape(-1, 6)
    kl_r   = (y_val_soft * np.log(np.clip(y_val_soft, 1e-7, 1) / np.clip(prob_r, 1e-7, 1))).sum(axis=1).mean()
    f1_r   = f1_score(y_val, prob_r.argmax(axis=1), average="macro", zero_division=0)
    stage_kl.append(kl_r)
    stage_f1.append(f1_r)

best_idx   = int(np.argmin(stage_kl))
BEST_ROUND = stage_rounds[best_idx]  # <- used for all "final" metrics/plots below
print(f"Best round (val KL)    : {BEST_ROUND}  (val KL={stage_kl[best_idx]:.4f})")


In [ ]:
# ============ Metrics (reported at the KL-optimal round) ============
prob_val = model.predict(dval, iteration_range=(0, BEST_ROUND)).reshape(-1, 6)
pred_val = prob_val.argmax(axis=1)

macro_f1  = f1_score(y_val, pred_val, average="macro", zero_division=0)
per_class = f1_score(y_val, pred_val, average=None,    zero_division=0)

kl_per_sample = (y_val_soft * np.log(np.clip(y_val_soft, 1e-7, 1) / np.clip(prob_val, 1e-7, 1))).sum(axis=1)
val_kl = kl_per_sample.mean()

# Also report metrics at the mlogloss-best round, so "loss-based" and "KL-based"
# selection can be compared side by side (they don't have to agree).
prob_val_loss = model.predict(dval, iteration_range=(0, model.best_iteration + 1)).reshape(-1, 6)
pred_val_loss = prob_val_loss.argmax(axis=1)
loss_f1 = f1_score(y_val, pred_val_loss, average="macro", zero_division=0)
loss_kl = (y_val_soft * np.log(np.clip(y_val_soft, 1e-7, 1) / np.clip(prob_val_loss, 1e-7, 1))).sum(axis=1).mean()

print(f"--- selected by mlogloss (round {model.best_iteration}) ---")
print(f"Val macro F1     : {loss_f1:.4f}")
print(f"Val KL divergence: {loss_kl:.4f}")
print(f"\n--- selected by val KL (round {BEST_ROUND}) ---")
print(f"Val macro F1     : {macro_f1:.4f}")
print(f"Val KL divergence: {val_kl:.4f}")
print(f"(random-guess baseline for 6 balanced classes: macro F1 \u2248 0.167)")
print("\nPer-class F1 (at the val-KL-selected round):")
for name, f in zip(CLASS_NAMES, per_class):
    print(f"  {name:<10} {f:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

rounds_all = range(len(evals_result["train"]["mlogloss"]))
axes[0, 0].plot(rounds_all, evals_result["train"]["mlogloss"], label="train")
axes[0, 0].plot(rounds_all, evals_result["val"]["mlogloss"],   label="val")
axes[0, 0].axvline(model.best_iteration, color="gray", linestyle="--",
                    label=f"mlogloss-best={model.best_iteration}")
axes[0, 0].axvline(BEST_ROUND, color="red", linestyle="--", label=f"KL-best={BEST_ROUND}")
axes[0, 0].set_xlabel("Round"); axes[0, 0].set_ylabel("mlogloss")
axes[0, 0].set_title("XGBoost mlogloss"); axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(stage_rounds, stage_kl, marker="o", color="darkorange")
axes[0, 1].axvline(BEST_ROUND, color="red", linestyle="--", label=f"best={BEST_ROUND}")
axes[0, 1].set_xlabel("Round"); axes[0, 1].set_ylabel("Val KL divergence")
axes[0, 1].set_title("Val KL divergence (every 5 rounds)"); axes[0, 1].legend(fontsize=8)

axes[1, 0].plot(stage_rounds, stage_f1, marker="o", color="seagreen")
axes[1, 0].axhline(1/6, color="gray", linestyle="--", linewidth=1, label="random guess")
axes[1, 0].axvline(BEST_ROUND, color="red", linestyle="--", label=f"best={BEST_ROUND}")
axes[1, 0].set_xlabel("Round"); axes[1, 0].set_ylabel("Val macro F1")
axes[1, 0].set_title("Val macro F1 (every 5 rounds)"); axes[1, 0].legend(fontsize=8)

axes[1, 1].bar(CLASS_NAMES, per_class, color="steelblue")
axes[1, 1].axhline(macro_f1, color="red", linestyle="--", label=f"macro F1 = {macro_f1:.3f}")
axes[1, 1].set_ylim(0, 1); axes[1, 1].set_ylabel("F1")
axes[1, 1].set_title(f"Per-class F1 (val, round {BEST_ROUND})"); axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("xgb_workshop_eval.png", dpi=150)
plt.show()


## Tweak & Compare (~20 min)

Pick 1-2 changes below, rerun the cell, and log your result in the shared sheet.

| Knob | Default | Try |
|---|---|---|
| `max_depth` | 6 | 3 / 9 |
| `num_boost_round` | 500 | 100 / 600 |
| feature subset | all bands | drop spectral power (mean/var/zero-crossing only) |

**Log your run:** What you changed | Val F1 | Val KL | one-line observation


In [ ]:
# ============ Tweak & Compare ============
TWEAK_MAX_DEPTH       = 6      # <- change me (try 3 or 9)
TWEAK_NUM_BOOST_ROUND = 500    # <- change me (try 100 or 600)
TWEAK_DROP_SPECTRAL   = False  # <- change me (try True to drop spectral-power features)


def build_features_tweak(df, cache_dir, drop_spectral=False, window_len=WINDOW_LEN):
    X, y_hard, y_soft = [], [], []
    for _, row in df.iterrows():
        eeg_id = int(row["eeg_id"])
        offset = int(row["eeg_label_offset_seconds"]) * FS

        eeg    = np.load(os.path.join(cache_dir, f"{eeg_id}.npy"))
        window = eeg[offset:offset + window_len]
        if len(window) < window_len:
            window = np.pad(window, ((0, window_len - len(window)), (0, 0)), mode="constant")
        window = np.nan_to_num(window, nan=0.0, posinf=0.0, neginf=0.0)

        feat_mean = window.mean(axis=0)
        feat_var  = window.var(axis=0)
        feat_zcr  = zero_crossing_rate(window)

        if drop_spectral:
            feats = np.concatenate([feat_mean, feat_var, feat_zcr])  # 3 x 19 = 57 features
        else:
            fft_power  = (np.abs(np.fft.rfft(window, axis=0)) ** 2) / len(window)
            feat_delta = fft_power[_DELTA].sum(axis=0)
            feat_theta = fft_power[_THETA].sum(axis=0)
            feat_alpha = fft_power[_ALPHA].sum(axis=0)
            feat_beta  = fft_power[_BETA].sum(axis=0)
            feats = np.concatenate([feat_mean, feat_var, feat_zcr,
                                     feat_delta, feat_theta, feat_alpha, feat_beta])  # 7 x 19 = 133

        soft = row[TARGETS].to_numpy(dtype=np.float32)
        soft = soft / soft.sum() if soft.sum() > 0 else np.full(6, 1 / 6)

        X.append(feats)
        y_hard.append(int(np.argmax(soft)))
        y_soft.append(soft)

    return (np.array(X, dtype=np.float32),
            np.array(y_hard),
            np.array(y_soft, dtype=np.float32))


Xt_train, yt_train, yt_train_soft = build_features_tweak(train_df, EEG_CACHE_DIR, TWEAK_DROP_SPECTRAL)
Xt_val,   yt_val,   yt_val_soft   = build_features_tweak(val_df,   EEG_CACHE_DIR, TWEAK_DROP_SPECTRAL)

tweak_params = dict(XGB_PARAMS)
tweak_params["max_depth"] = TWEAK_MAX_DEPTH

dtrain_t = xgb.DMatrix(Xt_train, label=yt_train)
dval_t   = xgb.DMatrix(Xt_val,   label=yt_val)

evals_result_t = {}
model_t = xgb.train(
    params                = tweak_params,
    dtrain                = dtrain_t,
    num_boost_round       = TWEAK_NUM_BOOST_ROUND,
    evals                 = [(dtrain_t, "train"), (dval_t, "val")],
    early_stopping_rounds = EARLY_STOPPING_ROUNDS,
    evals_result          = evals_result_t,
    verbose_eval          = False,
)

# stage val KL/F1 every 5 rounds, same as the baseline Metrics cell, so "best round"
# is picked the same way — otherwise this comparison isn't apples-to-apples
n_rounds_t   = len(evals_result_t["val"]["mlogloss"])
stage_every  = 5
stage_rounds_t = list(range(stage_every, n_rounds_t + 1, stage_every))
if not stage_rounds_t or stage_rounds_t[-1] != n_rounds_t:
    stage_rounds_t.append(n_rounds_t)

stage_kl_t = []
for r in stage_rounds_t:
    p = model_t.predict(dval_t, iteration_range=(0, r)).reshape(-1, 6)
    stage_kl_t.append((yt_val_soft * np.log(np.clip(yt_val_soft, 1e-7, 1) / np.clip(p, 1e-7, 1))).sum(axis=1).mean())

kl_best_round_t = stage_rounds_t[int(np.argmin(stage_kl_t))]


def _f1_kl_at(model, dval, y_val, y_val_soft, iteration_range):
    p = model.predict(dval, iteration_range=iteration_range).reshape(-1, 6)
    f1 = f1_score(y_val, p.argmax(axis=1), average="macro", zero_division=0)
    kl = (y_val_soft * np.log(np.clip(y_val_soft, 1e-7, 1) / np.clip(p, 1e-7, 1))).sum(axis=1).mean()
    return f1, kl


loss_f1_t, loss_kl_t = _f1_kl_at(model_t, dval_t, yt_val, yt_val_soft, (0, model_t.best_iteration + 1))
kl_f1_t,   kl_kl_t   = _f1_kl_at(model_t, dval_t, yt_val, yt_val_soft, (0, kl_best_round_t))

print(f"max_depth={TWEAK_MAX_DEPTH} | num_boost_round={TWEAK_NUM_BOOST_ROUND} | drop_spectral={TWEAK_DROP_SPECTRAL}")
print(f"--- selected by mlogloss (round {model_t.best_iteration}) ---")
print(f"Val macro F1 : {loss_f1_t:.4f}")
print(f"Val KL       : {loss_kl_t:.4f}")
print(f"\n--- selected by val KL (round {kl_best_round_t}) ---")
print(f"Val macro F1 : {kl_f1_t:.4f}")
print(f"Val KL       : {kl_kl_t:.4f}")
print(f"\n(baseline for comparison: mlogloss-best F1={loss_f1:.4f}/KL={loss_kl:.4f} | "
      f"val-KL-best F1={macro_f1:.4f}/KL={val_kl:.4f})")
print("-> log this row in the shared sheet: what you changed / F1 / KL / one-line observation")


### Done early? Extend (~10 min)

**Task:** add ONE new engineered feature to the feature vector and see if it moves val F1 / KL.
A good candidate: a per-channel **spectral peak frequency** (the frequency with the highest power
in each channel) — different information from the band-power features we already compute.

**Copy-paste prompt for Claude / ChatGPT:**

> Here is my feature extraction function and the function that builds a feature matrix from a
> dataframe: [paste both `extract_features` and `build_features` from earlier in this notebook].
> Add one new feature: for each channel, the frequency (in Hz) with the highest power in the FFT,
> using the `FS`, `WINDOW_LEN`, and `_FREQS` variables already defined in this notebook. Give me
> updated versions of both functions — call them `extract_features_with_peak_freq` and
> `build_features_with_peak_freq` — with the new feature concatenated onto the existing feature
> vector and everything else unchanged.
>
> Then, using these new functions, build train/val feature matrices, and train an XGBoost model
> with:
> ```
> XGB_PARAMS = {
>     "objective": "multi:softprob", "num_class": 6, "eval_metric": "mlogloss",
>     "tree_method": "hist", "max_depth": 6, "learning_rate": 0.05,
>     "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5, "seed": 42,
> }
> NUM_BOOST_ROUND = 500
> EARLY_STOPPING_ROUNDS = 30
> ```
> Print the resulting Val F1 and Val KL so I can compare against the baseline.

Paste the AI's code into the cell below, run it, then compare F1/KL against the baseline above.
If you get stuck for more than ~5 minutes, run the reference solution cell after this one.



In [ ]:
# ============ Your extended feature function goes here ============
# Paste the code Claude/ChatGPT gives you in response to the prompt above, e.g.:
#
# def extract_features_extended(window):
#     ...
#
# then rerun build_features_tweak-style code using your new function and compare
# F1/KL against the 133-feature baseline from the "Metrics" cell above.


### Reference solution (only look if you're stuck)


In [ ]:
# ============ Reference solution ============
def extract_features_with_peak_freq(window: np.ndarray) -> np.ndarray:
    """Same as extract_features(), plus one new feature: per-channel peak frequency."""
    window = np.nan_to_num(window, nan=0.0, posinf=0.0, neginf=0.0)

    feat_mean = window.mean(axis=0)
    feat_var  = window.var(axis=0)
    feat_zcr  = zero_crossing_rate(window)

    fft_power  = (np.abs(np.fft.rfft(window, axis=0)) ** 2) / len(window)
    feat_delta = fft_power[_DELTA].sum(axis=0)
    feat_theta = fft_power[_THETA].sum(axis=0)
    feat_alpha = fft_power[_ALPHA].sum(axis=0)
    feat_beta  = fft_power[_BETA].sum(axis=0)

    # NEW: peak frequency per channel
    peak_idx  = fft_power.argmax(axis=0)
    feat_peak = _FREQS[peak_idx]

    return np.concatenate([feat_mean, feat_var, feat_zcr,
                            feat_delta, feat_theta, feat_alpha, feat_beta,
                            feat_peak])  # 8 x 19 = 152 features


def build_features_with_peak_freq(df, cache_dir, window_len=WINDOW_LEN):
    X, y_hard, y_soft = [], [], []
    for _, row in df.iterrows():
        eeg_id = int(row["eeg_id"])
        offset = int(row["eeg_label_offset_seconds"]) * FS
        eeg    = np.load(os.path.join(cache_dir, f"{eeg_id}.npy"))
        window = eeg[offset:offset + window_len]
        if len(window) < window_len:
            window = np.pad(window, ((0, window_len - len(window)), (0, 0)), mode="constant")

        feats = extract_features_with_peak_freq(window)
        soft  = row[TARGETS].to_numpy(dtype=np.float32)
        soft  = soft / soft.sum() if soft.sum() > 0 else np.full(6, 1 / 6)

        X.append(feats); y_hard.append(int(np.argmax(soft))); y_soft.append(soft)
    return (np.array(X, dtype=np.float32), np.array(y_hard), np.array(y_soft, dtype=np.float32))


Xp_train, yp_train, _ = build_features_with_peak_freq(train_df, EEG_CACHE_DIR)
Xp_val,   yp_val,   yp_val_soft = build_features_with_peak_freq(val_df, EEG_CACHE_DIR)

dtrain_p = xgb.DMatrix(Xp_train, label=yp_train)
dval_p   = xgb.DMatrix(Xp_val,   label=yp_val)
model_p  = xgb.train(
    params=XGB_PARAMS, dtrain=dtrain_p, num_boost_round=NUM_BOOST_ROUND,
    evals=[(dtrain_p, "train"), (dval_p, "val")],
    early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose_eval=False,
)

# evaluated at its own mlogloss-best round — same criterion as the baseline's loss_f1/loss_kl
# numbers from the Metrics cell above, so this stays a fair, apples-to-apples comparison
prob_p = model_p.predict(dval_p, iteration_range=(0, model_p.best_iteration + 1)).reshape(-1, 6)
f1_p = f1_score(yp_val, prob_p.argmax(axis=1), average="macro", zero_division=0)
kl_p = (yp_val_soft * np.log(np.clip(yp_val_soft, 1e-7, 1) / np.clip(prob_p, 1e-7, 1))).sum(axis=1).mean()

print("133 baseline features + peak-freq (152 total), evaluated at its own mlogloss-best round:")
print(f"Val F1: {f1_p:.4f} | Val KL: {kl_p:.4f}")
print(f"(baseline at its own mlogloss-best, for a fair comparison: F1={loss_f1:.4f} | KL={loss_kl:.4f})")
